# Employee Attrition: Leakage-Aware Model Comparison

This notebook runs nested repeated cross-validation on the development split, selects one configuration using development PR-AUC, then evaluates that configuration once on an untouched stratified hold-out. All artifacts are written beneath `results/`. Set `QUICK_RUN = True` for a smoke run; use the full defaults for paper results.

In [ ]:
from pathlib import Path
import sys

WORKING_DIR = Path.cwd()
if WORKING_DIR.name == "notebooks":
    PROJECT_ROOT = WORKING_DIR.parent
elif (WORKING_DIR / "Project" / "src").is_dir():
    PROJECT_ROOT = WORKING_DIR / "Project"
else:
    PROJECT_ROOT = WORKING_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from src.employee_attrition import ExperimentConfig, run_experiment

DATA_PATH = PROJECT_ROOT / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
OUTPUT_DIR = PROJECT_ROOT / "results"
QUICK_RUN = False  # True: 2 folds x 1 repeat, 2 tuning trials
CONFIG = ExperimentConfig.quick() if QUICK_RUN else ExperimentConfig()
print(f"Dataset: {DATA_PATH}\nOutputs: {OUTPUT_DIR}\nConfig: {CONFIG}")

## Run the experiment

Install `requirements.txt` first. The source CSV is intentionally not bundled. The full protocol uses 5-fold x 10-repeat outer CV, inner randomized search on PR-AUC, and a final stratified 20% hold-out. The final hold-out is not used to compare or select configurations.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Place WA_Fn-UseC_-HR-Employee-Attrition.csv at {DATA_PATH}, or change DATA_PATH above.")
results = run_experiment(DATA_PATH, OUTPUT_DIR, CONFIG)

In [ ]:
results["summary"].sort_values("pr_auc_mean", ascending=False).head(15)

## Interpretation notes

The confidence intervals are conventional t intervals over repeated-CV fold scores; because repeated folds are dependent, treat them as descriptive rather than independent-sample inferential intervals. Wilcoxon comparisons are paired by identical outer folds. Strategy cost minima are reported from development CV; the final hold-out cost is shown only for the selected configuration to preserve the hold-out. The leakage demonstration and McNemar comparisons use development-only data and do not change model selection.